In [1]:
import os
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon GPU (MPS) aktif.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA GPU aktif.")
else:
    device = torch.device("cpu")
    print("CPU aktif.")

BASE_DIR = "."
IMAGE_DIR = os.path.join(BASE_DIR, "images")
MODEL_NAME = "tf_efficientnet_b0_ns"
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 0  # macOS multiprocessing hatasını engellemek için
EPOCHS = 10
LEARNING_RATE = 5e-4
CLASS_NAMES = ['healthy', 'multiple_diseases', 'rust', 'scab']
NUM_CLASSES = len(CLASS_NAMES)
BEST_MODEL_PATH = "best_plant_model.pth"

Apple Silicon GPU (MPS) aktif.


In [2]:
train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

train_df['label'] = np.argmax(train_df[CLASS_NAMES].values, axis=1)

def validate_and_clean_data(df, image_dir):
    valid_indices = []
    missing_count = 0
    corrupt_count = 0
    
    for idx, row in df.iterrows():
        img_path = os.path.join(image_dir, f"{row['image_id']}.jpg")
        if not os.path.exists(img_path):
            missing_count += 1
            continue
        if os.path.getsize(img_path) == 0:
            corrupt_count += 1
            continue
        valid_indices.append(idx)
        
    cleaned_df = df.loc[valid_indices].reset_index(drop=True)
    print(f"Toplam: {len(df)} | Geçerli: {len(cleaned_df)} | Eksik: {missing_count} | Bozuk: {corrupt_count}")
    return cleaned_df

train_df = validate_and_clean_data(train_df, IMAGE_DIR)
test_df = validate_and_clean_data(test_df, IMAGE_DIR)

train_data, val_data = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df['label']
)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(f"Eğitim Seti Boyutu: {len(train_data)}")
print(f"Doğrulama Seti Boyutu: {len(val_data)}")

Toplam: 1821 | Geçerli: 1821 | Eksik: 0 | Bozuk: 0
Toplam: 1821 | Geçerli: 1821 | Eksik: 0 | Bozuk: 0
Eğitim Seti Boyutu: 1456
Doğrulama Seti Boyutu: 365


In [3]:
train_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, border_mode=cv2.BORDER_REFLECT, p=0.7),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, min_holes=2, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class PlantDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None, is_test=False):
        self.df = df
        self.image_dir = image_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['image_id']
        img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
            
        if self.is_test:
            return image, img_id
        
        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label

train_loader = DataLoader(PlantDataset(train_data, IMAGE_DIR, transforms=train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(PlantDataset(val_data, IMAGE_DIR, transforms=val_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(PlantDataset(test_df, IMAGE_DIR, transforms=val_transforms, is_test=True), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

/opt/anaconda3/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/h_/4cvl86mn0lg7cfm1sm5bc48c0000gn/T/ipykernel_15831/2688408595.py:8: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16, min_holes=2, p=0.3),


In [4]:
class PlantClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=NUM_CLASSES, pretrained=True):
        super(PlantClassifier, self).__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        
    def forward(self, x):
        return self.backbone(x)

model = PlantClassifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES, pretrained=True).to(device)

# Azınlık sınıflarını dengelemek için Class Weights hesabı
class_counts = train_data['label'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = (class_weights / class_weights.sum()).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

/opt/anaconda3/lib/python3.13/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


In [5]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(dataloader, desc="Eğitim", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    return running_loss / total, correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds_probs, all_targets = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Doğrulama", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += labels.size(0)
            
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_preds_probs.extend(probs)
            all_targets.extend(labels.cpu().numpy())
            
    val_roc_auc = roc_auc_score(all_targets, all_preds_probs, multi_class='ovr')
    return running_loss / total, correct / total, val_roc_auc

In [6]:
best_roc_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_roc_auc = validate(model, val_loader, criterion, device)
    
    scheduler.step()
    elapsed = time.time() - start_time
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} ROC-AUC: {val_roc_auc:.4f}")
    
    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> En iyi model kaydedildi! (Val ROC-AUC: {best_roc_auc:.4f})")

print(f"\nEğitim tamamlandı! En yüksek Validation ROC-AUC: {best_roc_auc:.4f}")


--- Model Eğitimi Başlıyor ---


Epoch [01/10] (113.4s) | Train Loss: 1.3710 Acc: 0.6174 | Val Loss: 0.8317 Acc: 0.7945 ROC-AUC: 0.9166
  -> En iyi model kaydedildi! (Val ROC-AUC: 0.9166)


Epoch [02/10] (92.8s) | Train Loss: 0.7503 Acc: 0.7919 | Val Loss: 0.4609 Acc: 0.8822 ROC-AUC: 0.9652
  -> En iyi model kaydedildi! (Val ROC-AUC: 0.9652)


Epoch [03/10] (93.1s) | Train Loss: 0.5464 Acc: 0.8455 | Val Loss: 0.5410 Acc: 0.9096 ROC-AUC: 0.9629


Epoch [04/10] (94.6s) | Train Loss: 0.3551 Acc: 0.8922 | Val Loss: 0.4496 Acc: 0.9041 ROC-AUC: 0.9699
  -> En iyi model kaydedildi! (Val ROC-AUC: 0.9699)


...

Epoch [06/10] (103.7s) | Train Loss: 0.2397 Acc: 0.9293 | Val Loss: 0.4961 Acc: 0.8877 ROC-AUC: 0.9719


Epoch [07/10] (101.4s) | Train Loss: 0.1852 Acc: 0.9464 | Val Loss: 0.4943 Acc: 0.9205 ROC-AUC: 0.9699


Epoch [08/10] (113.9s) | Train Loss: 0.1576 Acc: 0.9485 | Val Loss: 0.4599 Acc: 0.9370 ROC-AUC: 0.9748
  -> En iyi model kaydedildi! (Val ROC-AUC: 0.9748)


Epoch [09/10] (120.7s) | Train Loss: 0.1544 Acc: 0.9560 | Val Loss: 0.4129 Acc: 0.9342 ROC-AUC: 0.9797
  -> En iyi model kaydedildi! (Val ROC-AUC: 0.9797)


Epoch [10/10] (114.5s) | Train Loss: 0.1448 Acc: 0.9595 | Val Loss: 0.4428 Acc: 0.9397 ROC-AUC: 0.9769

Eğitim tamamlandı! En yüksek Validation ROC-AUC: 0.9797


In [7]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_image_ids = []
test_predictions = []

with torch.no_grad():
    for images, img_ids in tqdm(test_loader, desc="Test Tahmini"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        
        test_predictions.extend(probs)
        test_image_ids.extend(img_ids)

submission_df = pd.DataFrame(test_predictions, columns=CLASS_NAMES)
submission_df.insert(0, 'image_id', test_image_ids)
submission_df.to_csv("submission.csv", index=False)

submission_df.head()

Test Tahmini: 100%|██████████| 114/114 [01:05<00:00,  1.75it/s]


,image_id,healthy,multiple_diseases,rust,scab
0,Test_0,4.590122e-06,1.082808e-02,9.891484e-01,1.891192e-05
1,Test_1,1.509082e-08,1.346637e-03,9.986532e-01,1.794009e-07
2,Test_2,2.183694e-07,2.413757e-03,2.245501e-08,9.975860e-01
3,Test_3,9.999970e-01,8.424795e-14,7.837515e-09,2.930343e-06
4,Test_4,1.626234e-09,3.752352e-05,9.999624e-01,3.929388e-10
